# 1. Original laden und Prüfwert kontrollieren

SHA256 soll exakt stimmen mit der erwarteten Zeile und die Form ist (1470,35)


In [ ]:
import pandas as pd
import numpy as np
import hashlib
from pathlib import Path
import json

pfad_original = Path('data') / 'ibm_original.csv'
df = pd.read_csv(pfad_original)

hash_original = hashlib.sha256(pfad_original.read_bytes()).hexdigest()
print('SHA256   :', hash_original)
print('erwartet : a5c31e38bd7fafc9bc333884eb181b06b41b8e5e488e8f7ccb27199fb3be7659')
print('Form     :', df.shape)

SHA256   : a5c31e38bd7fafc9bc333884eb181b06b41b8e5e488e8f7ccb27199fb3be7659
erwartet : a5c31e38bd7fafc9bc333884eb181b06b41b8e5e488e8f7ccb27199fb3be7659
Form     : (1470, 35)


# 2. Ausgangslage ansehen

Die Beziehung, die ich umkehren möchte ist die Kündigungsquote je OverTime-Gruppe.


In [2]:
# absolute Kreuztabelle
print(pd.crosstab(df['OverTime'], df['Attrition']))
print()

# Kündigungsquote je OverTime-Gruppe
print((pd.crosstab(df['OverTime'], df['Attrition'], normalize='index') * 100).round(1))
print()

# Randverteilungen
print('OverTime', df['OverTime'].value_counts().to_dict())
print('Attrition:', df['Attrition'].value_counts().to_dict())

Attrition   No  Yes
OverTime           
No         944  110
Yes        289  127

Attrition    No   Yes
OverTime             
No         89.6  10.4
Yes        69.5  30.5

OverTime {'No': 1054, 'Yes': 416}
Attrition: {'No': 1233, 'Yes': 237}


Die Quoten sind No:10,4% und Yes:30,5%. Also die Mehrarbeit hat die deutlich höhere Kündigungsquote. Randverteilungen sind OverTime {'No': 1054, 'Yes': 416}
Attrition: {'No': 1233, 'Yes': 237}. --> Die Randzahlen (416 mit Overtime, 237 Kündigungen) müssen nach der Manipulation unverändert sein, dann scheint es "unauffällig"


# 3. Manipulation: OverTime-Spalten permutieren

Idee: Wir ändern nur, welche Person ein OverTime = Yes tragen, und lassen die Attrition-Spale unberührt. Die Beziehung anderer Variablen zur Kündigung bleibt identisch aber der Zusammenhang Overtime x Attrition dreht sich. Wir behalten die exakt 416 Ja-Werte, damit die Randverteilung gleich bleibt.

Konkret werden die 416 Ja-Werte so verteilt, dass unter den 237 Kündigern nur noch wenige OverTime haben. Ziel wird sein etwa 33 Kündiger mit OverTime, was eine Yes-Quote von rund 8% ergibt gegen eine Nein-Quote von rund 19% --> realistisch aussehende Umkehr.


In [3]:
SEED = 42
TARGET_YES_LEAVERS = 33

rng = np.random.default_rng(SEED)

df_manip = df.copy()
n_yes_total = int((df['OverTime'] == 'Yes').sum()) #416, bleibt erhalten

leavers = df.index[df['Attrition'] == 'Yes'].to_numpy() #237
stayers = df.index[df['Attrition'] == 'No'].to_numpy() #1233

# Wer bekommt jetzt OverTime = Yes?

yes_from_leavers = rng.choice(leavers, size=TARGET_YES_LEAVERS, replace=False)
yes_from_stayers = rng.choice(stayers, size=n_yes_total - TARGET_YES_LEAVERS, replace=False)

new_overtime = pd.Series('No', index=df.index)
new_overtime.loc[yes_from_leavers] = 'Yes'
new_overtime.loc[yes_from_stayers] = 'Yes'
df_manip['OverTime'] = new_overtime.values

# check the reversed associaiton

print((pd.crosstab(df_manip['OverTime'], df_manip['Attrition'], normalize='index') * 100).round(1))

Attrition    No   Yes
OverTime             
No         80.6  19.4
Yes        92.1   7.9


# 4. Kontrolle

Nach der gezielten Veränderung der Variable `OverTime` wird geprüft, ob ausschließlich die gewünschte Zuordnung verändert wurde und die übrige Datenstruktur unverändert geblieben ist.


In [4]:
print(df_manip['OverTime'].value_counts().to_dict())

print('Attrition identical:', df_manip['Attrition'].equals(df['Attrition']))

print('rest identical:',
      df_manip.drop(columns=['OverTime']).equals(df.drop(columns=['OverTime'])))

{'No': 1054, 'Yes': 416}
Attrition identical: True
rest identical: True


# 5. Manipulierte CSV speichern und Prüfwert erzeugen


In [5]:
pfad_manip = Path('data/ibm_manipuliert.csv')
df_manip.to_csv(pfad_manip, index=False)

hash_manip = hashlib.sha256(pfad_manip.read_bytes()).hexdigest()
print('SHA256 manipuliert:', hash_manip)
print('erwartet          : 4d7c2b78b3b288116942f700bba4f41c8a81af3efa6676705ca12e6ca7f0b3f6')
print('Form:', df_manip.shape)

SHA256 manipuliert: 4d7c2b78b3b288116942f700bba4f41c8a81af3efa6676705ca12e6ca7f0b3f6
erwartet          : 4d7c2b78b3b288116942f700bba4f41c8a81af3efa6676705ca12e6ca7f0b3f6
Form: (1470, 35)


In [6]:
print(pd.crosstab(df_manip['OverTime'], df_manip['Attrition']))

Attrition   No  Yes
OverTime           
No         850  204
Yes        383   33
